# Notebook 1: build the lake and the first two data products

Builds the first two divisions of the airline data mesh from public data and checks the Dec 2022 case.

**Flow:** raw files land in the **lake** (Google Drive) → each division's data is published as a table in the **warehouse** (BigQuery, one dataset per division) → we query the warehouse to check the case.

**Before you start:** create a free Google Cloud project (no billing needed for the BigQuery sandbox) and copy its **project ID**.

**What the check looks for:** airports with clear weather but heavy cancellations, and whether cancellations stayed high after the weather cleared.

## Cell 1: setup
Installs libraries, connects Google Drive (the lake) and signs in to Google (the warehouse). Edit `PROJECT`, and optionally `MONTHS` and `CARRIER`.

In [ ]:
# Install what this notebook needs. The BigQuery client is already in Colab.
!pip -q install pandas pyarrow requests

# Drive is the "lake": raw and landed files live there and survive between runs.
# auth gives this notebook permission to write to BigQuery, the "warehouse".
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

from pathlib import Path
from google.cloud import bigquery

# Your Google Cloud project ID (not its display name). The project picker at
# the top of the Cloud console shows it.
PROJECT = 'YOUR_GCP_PROJECT_ID'
bq = bigquery.Client(project=PROJECT)

# Where the lake lives on Drive.
ROOT = Path('/content/drive/MyDrive/airline-data-mesh/lake')
ROOT.mkdir(parents=True, exist_ok=True)

# Which BTS months to load, and which carrier to replay ('WN' = Southwest).
# Cell 4's weather window is hardcoded to Dec 2022; change it if you change this.
MONTHS = [(2022, 12)]
CARRIER = 'WN'

## Cell 2: land the raw BTS files in the lake
Downloads each month of the U.S. DOT Bureau of Transportation Statistics On-Time Performance file (all carriers, one row per flight) and stores the CSV as received. Months already on Drive are skipped, so re-running costs nothing. The file URL pattern is from memory; if it returns a 404, it may have changed.

In [ ]:
import requests, zipfile, io

raw = ROOT / '_raw'
raw.mkdir(exist_ok=True)

for y, m in MONTHS:
    f = raw / f'bts_{y}_{m:02d}.csv'
    if f.exists():
        continue  # already landed on an earlier run
    # The URL month is not zero-padded (..._2022_1, not ..._2022_01).
    url = ('https://transtats.bts.gov/PREZIP/'
           f'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{y}_{m}.zip')
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    # Each zip holds one CSV; keep it exactly as received (the raw layer).
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        f.write_bytes(z.read(z.namelist()[0]))
    print('saved', f.name, f.stat().st_size // 1_000_000, 'MB')

## Cell 3: build the Network operations data product
Keeps one carrier and the columns we need, renames them to the contract's names (`domains/network_ops/contract.yaml`) and saves Parquet. Parquet is the landing format: compact, typed, and loaded natively by BigQuery.

In [ ]:
import pandas as pd

# BTS column name -> the name in our contract.
RENAME = {
    'FlightDate': 'flight_date', 'Reporting_Airline': 'carrier',
    'Tail_Number': 'tail_number', 'Flight_Number_Reporting_Airline': 'flight_number',
    'Origin': 'origin', 'Dest': 'dest', 'CRSDepTime': 'crs_dep_time',
    'DepDelay': 'dep_delay_min', 'Cancelled': 'cancelled',
    'CancellationCode': 'cancel_code', 'Diverted': 'diverted',
    'CarrierDelay': 'carrier_delay_min', 'WeatherDelay': 'weather_delay_min',
    'NASDelay': 'nas_delay_min', 'SecurityDelay': 'security_delay_min',
    'LateAircraftDelay': 'late_aircraft_delay_min'}

parts = []
for f in sorted(raw.glob('bts_*.csv')):
    # Read only the columns we need, then keep only our carrier's flights.
    df = pd.read_csv(f, usecols=list(RENAME), low_memory=False)
    parts.append(df[df.Reporting_Airline == CARRIER])

fl = pd.concat(parts, ignore_index=True).rename(columns=RENAME)
fl['flight_date'] = pd.to_datetime(fl['flight_date']).dt.date  # a date, not a timestamp

out = ROOT / 'flight_ops'
out.mkdir(exist_ok=True)
fl.to_parquet(out / 'flights.parquet', index=False)
print(len(fl), 'flights,', int(fl.cancelled.sum()), 'cancelled')

## Cell 4: build the Weather data product
Downloads hourly airport weather observations (NOAA ASOS, served by the Iowa Environmental Mesonet) for the 25 busiest airports in one request, then saves Parquet. One request keeps us under the service's rate limit; the loop retries if it still says "too many requests". Airports outside the top 25 are left out, which can hide a clear-weather airport with heavy cancellations; widen `head(25)` to check.

In [ ]:
import time

# The 25 airports with the most flights in the carrier's data.
airports = fl.origin.value_counts().head(25).index.tolist()

# station=[...] sends all airports in a single request.
# Times are UTC. 'M' marks missing values; we turn those into nulls below.
p = dict(station=airports, data=['tmpf', 'vsby', 'sknt', 'wxcodes'],
         year1=2022, month1=12, day1=1, year2=2023, month2=1, day2=1,
         tz='Etc/UTC', format='onlycomma', latlon='no', missing='M',
         trace='T', direct='no', report_type=[3, 4])

for attempt in range(5):
    r = requests.get('https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py',
                     params=p, timeout=300)
    if r.status_code == 429:  # rate limited: wait longer each time, then retry
        wait = 30 * (attempt + 1)
        print(f'rate limited, waiting {wait}s')
        time.sleep(wait)
        continue
    r.raise_for_status()
    break
else:
    raise RuntimeError('still rate limited; wait a few minutes and rerun this cell')

wx = pd.read_csv(io.StringIO(r.text), na_values=['M'])
wx['valid'] = pd.to_datetime(wx['valid'])

wxdir = ROOT / 'weather'
wxdir.mkdir(exist_ok=True)
# Microsecond timestamps: BigQuery loads pandas' default nanosecond timestamps
# as plain integers, which breaks date functions later.
wx.to_parquet(wxdir / 'observations.parquet', index=False,
              coerce_timestamps='us', allow_truncated_timestamps=True)
print(len(wx), 'observations,', wx.station.nunique(), 'airports')

## Cell 5: publish both products to the warehouse
Loads each Parquet file into its own BigQuery dataset: `network_ops.flights` and `weather.observations`. This is the "one dataset per division" shape the mesh uses. `WRITE_TRUNCATE` replaces the table on each run, so re-running does not create duplicates.

In [ ]:
def publish(dataset, table, parquet_path):
    """Load one Parquet file into BigQuery as <project>.<dataset>.<table>."""
    bq.create_dataset(bigquery.Dataset(f'{PROJECT}.{dataset}'), exists_ok=True)
    cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition='WRITE_TRUNCATE')
    with open(parquet_path, 'rb') as f:
        bq.load_table_from_file(f, f'{PROJECT}.{dataset}.{table}',
                                job_config=cfg).result()

publish('network_ops', 'flights', out / 'flights.parquet')
publish('weather', 'observations', wxdir / 'observations.parquet')

# Sanity check: how many flights landed.
print(bq.query(f'SELECT count(*) AS n FROM `{PROJECT}.network_ops.flights`')
      .to_dataframe())

## Cell 6: the case check, by airport
For Dec 21 to 31, joins each airport's cancellations to its weather and prints one row per airport. The second table lists airports with almost no bad weather but a cancellation rate above 20%.

Caveats: weather days are UTC and flight days are local, so the edges are approximate, and "bad weather" is a crude rule. BTS cancellation codes are self-reported by the carrier (A carrier, B weather, C air traffic system).

In [ ]:
sql = f"""
-- Per airport and day: the share of weather observations that look bad.
-- 'Bad' = snow, freezing, thunderstorm, ice codes, visibility under 3 miles,
-- or wind of 35 knots or more.
WITH wx AS (
  SELECT station, DATE(valid) AS day,
    AVG(CAST(REGEXP_CONTAINS(IFNULL(wxcodes, ''), r'SN|FZ|TS|PL|GR|IC')
             OR vsby < 3 OR sknt >= 35 AS INT64)) AS bad_wx_share
  FROM `{PROJECT}.weather.observations`
  GROUP BY station, day),
-- Per airport and day: flights, cancellations, and the carrier's cause codes.
fl AS (
  SELECT origin, flight_date AS day, COUNT(*) AS flights,
    SUM(cancelled) AS cancelled,
    COUNTIF(cancel_code = 'A') AS code_a,
    COUNTIF(cancel_code = 'B') AS code_b,
    COUNTIF(cancel_code = 'C') AS code_c
  FROM `{PROJECT}.network_ops.flights`
  WHERE flight_date BETWEEN '2022-12-21' AND '2022-12-31'
  GROUP BY origin, day)
SELECT fl.origin, SUM(flights) AS flights, SUM(cancelled) AS cancelled,
  ROUND(SUM(cancelled) / SUM(flights), 2) AS cancel_rate,
  SUM(code_a) AS code_a, SUM(code_b) AS code_b, SUM(code_c) AS code_c,
  ROUND(AVG(bad_wx_share), 2) AS bad_wx_share
FROM fl JOIN wx ON wx.station = fl.origin AND wx.day = fl.day
GROUP BY fl.origin ORDER BY cancel_rate DESC"""

t = bq.query(sql).to_dataframe()
print(t.to_string(index=False))
print('\nClear weather but heavy cancellations:')
print(t[(t.bad_wx_share < 0.05) & (t.cancel_rate > 0.2)].to_string(index=False))

## Cell 7: the same check, day by day
Shows cancellation rate and bad-weather share per day at five airports, from before the event to after it. Look for airports where cancellations keep climbing while their weather share stays at zero, and for days where all airports move together regardless of local weather.

In [ ]:
sql = f"""
WITH wx AS (
  SELECT station, DATE(valid) AS day,
    AVG(CAST(REGEXP_CONTAINS(IFNULL(wxcodes, ''), r'SN|FZ|TS|PL|GR|IC')
             OR vsby < 3 OR sknt >= 35 AS INT64)) AS bad_wx_share
  FROM `{PROJECT}.weather.observations`
  GROUP BY station, day),
fl AS (
  SELECT origin, flight_date AS day, COUNT(*) AS flights, SUM(cancelled) AS cancelled
  FROM `{PROJECT}.network_ops.flights`
  -- Edit this list to look at other airports.
  WHERE origin IN ('LAS', 'DAL', 'PHX', 'DEN', 'MDW')
    AND flight_date BETWEEN '2022-12-18' AND '2022-12-31'
  GROUP BY origin, day)
SELECT fl.origin, fl.day, flights, cancelled,
  ROUND(cancelled / flights, 2) AS cancel_rate,
  ROUND(bad_wx_share, 2) AS bad_wx_share
FROM fl JOIN wx ON wx.station = fl.origin AND wx.day = fl.day
ORDER BY fl.origin, fl.day"""

d = bq.query(sql).to_dataframe()
# One column per airport, one row per day.
print(d.pivot(index='day', columns='origin',
              values=['cancel_rate', 'bad_wx_share']).to_string())

## Cell 8: did the schedule shrink, and how many aircraft flew?
Network-wide, per day: scheduled flights, cancellations, and the number of distinct aircraft (tail numbers) that actually flew. If scheduled flights stay flat while aircraft flying falls, the network lost capacity instead of cutting its schedule.

In [ ]:
sql = f"""
SELECT flight_date AS day, COUNT(*) AS scheduled, SUM(cancelled) AS cancelled,
  -- Aircraft that flew at least one flight that day.
  COUNT(DISTINCT IF(cancelled = 0, tail_number, NULL)) AS tails_flying
FROM `{PROJECT}.network_ops.flights`
WHERE flight_date BETWEEN '2022-12-18' AND '2022-12-31'
GROUP BY day ORDER BY day"""
print(bq.query(sql).to_dataframe().to_string(index=False))